# TrashScan no VS Code / Jupyter Local

Este notebook foi adaptado para rodar **dentro do repositório clonado** do projeto `TrashScan`, no VS Code ou em outro ambiente Jupyter local.

## Premissas desta versão

- Você fará `git clone` do repositório e abrirá este notebook **de dentro dele**
- Os scripts antes baixados por `gdown` agora serão usados **diretamente do repositório**
- Os datasets **TACO** e **Roboflow** continuam sendo baixados externamente, mas serão integrados ao pipeline local
- O notebook foi reorganizado para:
  - evitar dependências de `google.colab`
  - evitar caminhos `/content/...`
  - explicar cada etapa com células markdown
  - validar estrutura, caminhos e arquivos antes de treinar

## Estrutura esperada

Este notebook assume uma estrutura semelhante a:

```text
TrashScan/
├── data/
├── env/
├── eval/
├── notebooks/
├── train/
├── utils/
└── ...
```

## Como executar

A ordem sugerida é:

1. Preparar o ambiente
2. Validar o repositório e os scripts
3. Configurar caminhos
4. Baixar / localizar datasets externos
5. Fazer merge e preprocessamento
6. Validar artefatos
7. Treinar
8. Avaliar

## 1) Descoberta automática do diretório do repositório

A célula abaixo tenta localizar a raiz do repositório `TrashScan` automaticamente, mesmo se o notebook estiver em `notebooks/`.

Se necessário, você poderá ajustar a variável `REPO_ROOT` manualmente na próxima célula.

In [1]:

from pathlib import Path
import os

# Tenta descobrir a raiz do repositório
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)

REPO_ROOT = None
for p in candidates:
    if (p / "data").exists() and (p / "train").exists() and (p / "eval").exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

NOTEBOOK_DIR = cwd
DATA_DIR = REPO_ROOT / "data"
TRAIN_DIR = REPO_ROOT / "train" / "paths"
EVAL_DIR = REPO_ROOT / "eval"
UTILS_DIR = REPO_ROOT / "utils"
ENV_DIR = REPO_ROOT / "env"

print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("REPO_ROOT    =", REPO_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("TRAIN_DIR    =", TRAIN_DIR)
print("EVAL_DIR     =", EVAL_DIR)
print("UTILS_DIR    =", UTILS_DIR)

NOTEBOOK_DIR = /home/lucas/TrashScan/notebooks
REPO_ROOT    = /home/lucas/TrashScan
DATA_DIR     = /home/lucas/TrashScan/data
TRAIN_DIR    = /home/lucas/TrashScan/train/paths
EVAL_DIR     = /home/lucas/TrashScan/eval
UTILS_DIR    = /home/lucas/TrashScan/utils


## 2) Configuração de caminhos locais

Ajuste aqui apenas se você quiser guardar datasets e saídas em outro lugar.

Por padrão:
- os datasets externos ficam em `TrashScan/external_datasets`
- os artefatos processados ficam em `TrashScan/processed_4cls`
- os pesos/copias locais ficam em `TrashScan/path_A_weights`
- as saídas de treino ficam em `TrashScan/runs/path_B`
- os resultados de avaliação ficam em `TrashScan/results_4cls`

In [2]:

from pathlib import Path
import os

from pathlib import Path

PARENT_DIR = REPO_ROOT.parent
EXTERNAL_DIR = PARENT_DIR / "external_datasets"
TACO_DIR = PARENT_DIR / "TACO"
PROCESSED_DIR = PARENT_DIR / "processed_4cls"
PATH_A_WEIGHTS_DIR = PARENT_DIR / "path_A_weights"
RUNS_PATH_B_DIR = PARENT_DIR / "runs" / "path_B"
RESULTS_DIR = PARENT_DIR / "results_4cls"

for p in [EXTERNAL_DIR, PROCESSED_DIR, PATH_A_WEIGHTS_DIR, RUNS_PATH_B_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("EXTERNAL_DIR       =", EXTERNAL_DIR)
print("TACO_DIR           =", TACO_DIR)
print("PROCESSED_DIR      =", PROCESSED_DIR)
print("PATH_A_WEIGHTS_DIR =", PATH_A_WEIGHTS_DIR)
print("RUNS_PATH_B_DIR    =", RUNS_PATH_B_DIR)
print("RESULTS_DIR        =", RESULTS_DIR)

EXTERNAL_DIR       = /home/lucas/external_datasets
TACO_DIR           = /home/lucas/TACO
PROCESSED_DIR      = /home/lucas/processed_4cls
PATH_A_WEIGHTS_DIR = /home/lucas/path_A_weights
RUNS_PATH_B_DIR    = /home/lucas/runs/path_B
RESULTS_DIR        = /home/lucas/results_4cls


## 3) Verificação do conteúdo do repositório

Aqui nós garantimos que os scripts usados no pipeline realmente existem no clone local.

In [3]:

required_paths = [
    DATA_DIR / "merge_datasets.py",
    DATA_DIR / "preprocess.py",
    TRAIN_DIR / "train_path_B.py",
    EVAL_DIR / "evaluate.py",
    EVAL_DIR / "evaluate_path_B_combined.py",
    UTILS_DIR / "validate_all.py",
]

missing = [str(p) for p in required_paths if not p.exists()]

if missing:
    print("Arquivos ausentes:")
    for m in missing:
        print(" -", m)
    raise FileNotFoundError("Há scripts esperados que não foram encontrados no repositório.")
else:
    print("Tudo certo. Scripts essenciais encontrados.")

Tudo certo. Scripts essenciais encontrados.


## 4) Preparação do ambiente Python

Você tem duas opções:

### Opção A — usar `conda` com o YAML do projeto
No terminal, fora do notebook:

```bash
conda env create -f env/environment_AB.yml
conda activate <nome-do-ambiente>
```

### Opção B — instalar pelo notebook
A célula abaixo faz uma instalação local por `pip`.

> Observação: para projetos maiores, normalmente a opção com `conda` é mais estável.

In [4]:

import sys
print(sys.executable)

/home/lucas/.venv/bin/python


In [ ]:

# Descomente se quiser instalar pelo notebook
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install ultralytics==8.3.2 timm==1.0.9 roboflow gdown pycocotools

## 5) Utilitários de execução

As próximas células usam `subprocess` para executar scripts do repositório de forma mais previsível no VS Code/Jupyter local.

In [ ]:

import subprocess
import shlex
import os
from pathlib import Path

def run_cmd(cmd, cwd=PARENT_DIR, env=None):
    if isinstance(cmd, str):
        print("$", cmd)
        result = subprocess.run(cmd, cwd=str(cwd), shell=True, env=env, check=True)
    else:
        print("$", " ".join(shlex.quote(str(x)) for x in cmd))
        result = subprocess.run([str(x) for x in cmd], cwd=str(cwd), env=env, check=True)
    return result

## 6) Verificação de GPU / CPU

O notebook original assumia GPU do Colab. Aqui a célula detecta se há CUDA disponível e define parâmetros iniciais.

Você ainda pode sobrescrever `DEVICE` ou `BATCH` manualmente se quiser.

In [ ]:

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    DEVICE = "0"
    if "A100" in gpu_name:
        BATCH = 64
    elif "V100" in gpu_name:
        BATCH = 32
    else:
        BATCH = 16
else:
    gpu_name = "cpu"
    DEVICE = "cpu"
    BATCH = 8

print("Dispositivo:", gpu_name)
print("DEVICE     :", DEVICE)
print("BATCH      :", BATCH)

## 7) Pesos locais do detector

Como agora você vai usar o repositório clonado, esta versão prioriza **artefatos já presentes no clone** em vez de `gdown`.

A ordem sugerida é:
1. procurar pesos em `YOLO_on_TACO_deprecated/runs/detect/...`
2. copiar para `path_A_weights/`
3. opcionalmente tentar extrair `yolov8m/*.zip` se você quiser reaproveitar esses arquivos

> Se você já tiver um `best.pt` definitivo, basta colocá-lo em `path_A_weights/yolov8m_best.pt`.

In [ ]:

from pathlib import Path
import shutil

candidate_best = [
    REPO_ROOT / "path_A_weights" / "yolov8m_best.pt",
    REPO_ROOT / "YOLO_on_TACO_deprecated" / "runs" / "detect" / "train" / "weights" / "best.pt",
]

found_best = None
for c in candidate_best:
    if c.exists():
        found_best = c
        break

if found_best is not None:
    target = PATH_A_WEIGHTS_DIR / "yolov8m_best.pt"
    if found_best.resolve() != target.resolve():
        shutil.copy2(found_best, target)
    print("Peso detector disponível em:", target)
else:
    print("Nenhum best.pt encontrado automaticamente.")
    print("Coloque manualmente um peso em:", PATH_A_WEIGHTS_DIR / "yolov8m_best.pt")

### Extração opcional dos arquivos em `yolov8m/`

Use esta célula **somente** se você realmente precisar extrair os arquivos multipart (`.z01`, `.z02`, `.z03`, `.zip`) presentes no repositório.

Ela usa `7z`, então seu sistema precisa ter `7z` instalado e acessível no terminal.

In [ ]:

# EXTRAÇÃO OPCIONAL — descomente se quiser usar
# archive_dir = REPO_ROOT / "yolov8m"
# if archive_dir.exists():
#     run_cmd(f'7z x "{archive_dir / "yolov8m.zip"}" -o"{archive_dir / "extracted"}"')
# else:
#     print("Diretório yolov8m/ não encontrado.")

## 8) Download do TACO

Como você comentou que o TACO continuará vindo de fora do repositório, esta etapa foi deixada separada.

Há duas estratégias:

### Estratégia A — usar o script do repo
Se `data/download_external_datasets.py` já baixa o TACO no formato correto, prefira usá-lo.

### Estratégia B — baixar manualmente
Baixe o TACO para a pasta `TACO/` na raiz do repositório.

A célula seguinte mostra ajuda do script, para você confirmar os argumentos corretos.

In [ ]:

# Ajuda do downloader do repositório
downloader = DATA_DIR / "download_external_datasets.py"
if downloader.exists():
    run_cmd([sys.executable, str(downloader), "--help"])
else:
    print("Script de download externo não encontrado.")

### Download do TACO usando o script do projeto

Descomente e ajuste se o `--help` acima confirmar que esse é o caminho correto no seu repo.

In [ ]:

# EXEMPLO — ajuste conforme a interface real do script
# run_cmd([
#     sys.executable, str(DATA_DIR / "download_external_datasets.py"),
#     "--output_root", str(TACO_DIR)
# ])

### Verificação manual do TACO

Se você baixar o TACO por fora, rode a célula abaixo para confirmar que a pasta existe antes de seguir.

In [ ]:

print("TACO_DIR existe?", TACO_DIR.exists())
if TACO_DIR.exists():
    for p in list(TACO_DIR.iterdir())[:10]:
        print(" -", p.name)

## 9) Download do dataset extra via Roboflow

No notebook original havia uma chave de API hardcoded. Aqui isso foi trocado por variável de ambiente.

Antes de rodar a célula abaixo, defina no terminal:

```bash
export ROBOFLOW_API_KEY="SUA_CHAVE"
```

No Windows PowerShell:

```powershell
$env:ROBOFLOW_API_KEY="SUA_CHAVE"
```

> Recomendação: use uma chave nova, especialmente se a antiga já apareceu em notebook antigo.

In [ ]:

import os
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "")
print("ROBOFLOW_API_KEY carregada?", bool(ROBOFLOW_API_KEY))

In [ ]:

# Download Roboflow — descomente quando a variável de ambiente estiver definida
# from roboflow import Roboflow
#
# rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# project = rf.workspace("sadis-workspace").project("taco-dataset-ql1ng-atu1k")
# version = project.version(3)
# dataset = version.download("coco")
#
# print("Dataset baixado em:", dataset.location)

### Organização do dataset extra no layout esperado

No notebook antigo, os datasets do Roboflow eram movidos para:

```text
external_datasets/coco_format/
```

A célula abaixo cria esse destino e serve como base para mover os diretórios baixados.

In [ ]:

COCO_EXTERNAL_DIR = EXTERNAL_DIR / "coco_format"
COCO_EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)
print("Destino esperado:", COCO_EXTERNAL_DIR)

In [ ]:

# Ajuste os nomes conforme o Roboflow gerar as pastas no seu ambiente
# possible_rf_dirs = [
#     REPO_ROOT / "TACO-dataset-1",
#     REPO_ROOT / "TACO-dataset-2",
#     REPO_ROOT / "TACO-dataset-3",
# ]
#
# import shutil
# for d in possible_rf_dirs:
#     if d.exists():
#         target = COCO_EXTERNAL_DIR / d.name
#         if target.exists():
#             shutil.rmtree(target)
#         shutil.move(str(d), str(target))
#         print("Movido:", d, "->", target)

## 10) Merge dos datasets

Aqui começa o pipeline do projeto usando os scripts do repositório local.

Este passo deve combinar:
- o TACO
- os datasets externos já baixados
- a estrutura de saída usada no Path B

In [ ]:

merge_script = DATA_DIR / "merge_datasets.py"

run_cmd([
    sys.executable, str(merge_script),
    "--taco_root", str(TACO_DIR),
    "--external_root", str(EXTERNAL_DIR),
    "--output_root", str(PROCESSED_DIR),
    "--skip_preprocess",
])

## 11) Pré-processamento

No notebook antigo o pré-processamento rodava sobre `processed_4cls/merged_data` e gerava os artefatos para o Path B.

Se o script consumir muita RAM no seu ambiente, considere rodar em máquina com mais memória ou adaptar o script do projeto.

In [ ]:

preprocess_script = DATA_DIR / "preprocess.py"

run_cmd([
    sys.executable, str(preprocess_script),
    "--taco_root", str(PROCESSED_DIR / "merged_data"),
    "--output_root", str(PROCESSED_DIR),
    "--path", "B",
])

## 12) Validação dos artefatos processados

Esta etapa ajuda a detectar mais cedo problemas de caminhos, arquivos faltando ou datasets inconsistentes.

In [ ]:

validate_script = UTILS_DIR / "validate_all.py"

run_cmd([
    sys.executable, str(validate_script),
    "--check_processed",
    "--processed_dir", str(PROCESSED_DIR),
])

## 13) Conferência do peso do detector antes do treino

O `train_path_B.py` precisa de um peso de detector válido.

A célula abaixo valida a presença do arquivo esperado.

In [ ]:

DETECTOR_WEIGHTS = PATH_A_WEIGHTS_DIR / "yolov8m_best.pt"
print("DETECTOR_WEIGHTS =", DETECTOR_WEIGHTS)

if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Peso do detector não encontrado em {DETECTOR_WEIGHTS}. "
        "Coloque o arquivo manualmente ou copie de outro local do repositório."
    )

## 14) Treinamento do Path B

No seu repositório atual, o script correto é:

```text
train/paths/train_path_B.py
```

Você pode treinar um classificador por vez, como no notebook original, ou passar múltiplos nomes se o script aceitar isso.

### Modelo 1 — `resnet50`

In [ ]:

train_script = TRAIN_DIR / "train_path_B.py"

run_cmd([
    sys.executable, str(train_script),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--crops_dir", str(PROCESSED_DIR),
    "--output", str(RUNS_PATH_B_DIR),
    "--classifiers", "resnet50",
    "--epochs", "50",
    "--device", str(DEVICE),
])

### Avaliação do Modelo 1

In [ ]:

eval_script = EVAL_DIR / "evaluate.py"

run_cmd([
    sys.executable, str(eval_script),
    "--path", "B",
    "--runs_dir", str(RUNS_PATH_B_DIR),
    "--output", str(RESULTS_DIR),
])

### Modelo 2 — `vit_b16_scratch`

In [ ]:

# Descomente para treinar o segundo modelo
# run_cmd([
#     sys.executable, str(train_script),
#     "--detector_weights", str(DETECTOR_WEIGHTS),
#     "--crops_dir", str(PROCESSED_DIR),
#     "--output", str(RUNS_PATH_B_DIR),
#     "--classifiers", "vit_b16_scratch",
#     "--epochs", "50",
#     "--device", str(DEVICE),
# ])

### Modelo 3 — `vit_b16_imagenet`

In [ ]:

# Descomente para treinar o terceiro modelo
# run_cmd([
#     sys.executable, str(train_script),
#     "--detector_weights", str(DETECTOR_WEIGHTS),
#     "--crops_dir", str(PROCESSED_DIR),
#     "--output", str(RUNS_PATH_B_DIR),
#     "--classifiers", "vit_b16_imagenet",
#     "--epochs", "50",
#     "--device", str(DEVICE),
# ])

## 15) Avaliação combinada

No notebook antigo era usado `evaluate_path_B_combined2.py`, mas **no seu repositório atual** o arquivo listado é:

```text
eval/evaluate_path_B_combined.py
```

Então a célula abaixo já usa esse nome atualizado.

In [ ]:

combined_eval_script = EVAL_DIR / "evaluate_path_B_combined.py"

run_cmd([
    sys.executable, str(combined_eval_script),
    "--detector_weights", str(DETECTOR_WEIGHTS),
    "--classifier_dir", str(RUNS_PATH_B_DIR / "path_B"),
    "--classifiers", "resnet50", "vit_b16_scratch", "vit_b16_imagenet",
    "--data_yaml", str(PROCESSED_DIR / "dataset_path_B.yaml"),
    "--output", str(RESULTS_DIR),
])

## 16) Inspeção final dos resultados

Use as células abaixo para conferir rapidamente o que foi gerado.

In [ ]:

for p in [RUNS_PATH_B_DIR, RESULTS_DIR, PROCESSED_DIR]:
    print("\n==", p, "==")
    if p.exists():
        items = list(p.iterdir())[:20]
        for x in items:
            print(" -", x.name)
    else:
        print("Diretório não existe.")

## Observações finais

### O que mudou em relação ao notebook do Colab
- removido `google.colab.drive.mount(...)`
- removidos caminhos `/content/...`
- removidos trechos que dependiam de copiar resultados para o Google Drive
- scripts agora são executados a partir do **clone local**
- pesos e datasets locais são tratados como artefatos do repositório / da máquina

### O que você ainda precisa conferir no primeiro uso
- a interface exata de `data/download_external_datasets.py`
- o nome final das pastas baixadas pelo Roboflow
- se `train_path_B.py` aceita exatamente esses argumentos no seu clone atual
- se `evaluate_path_B_combined.py` espera `--classifier_dir runs/path_B/path_B` ou apenas `runs/path_B`

Se algum desses scripts tiver mudado desde o notebook antigo, a adaptação de caminhos já está pronta; aí basta ajustar os argumentos pontuais.